# Spark 4.1 Real-time Mode — Kafka -> stateless transform -> Kafka

This notebook demonstrates the **Real-time Mode** streaming trigger introduced in
**Apache Spark 4.1.2**. Real-time Mode launches long-running tasks (one per Kafka
partition) so records flow through `source -> transformations -> sink` continuously,
targeting millisecond-scale latency instead of the ~100 ms micro-batch floor.

The demo:
1. Produces ~10,000 events/sec into the `events-input` Kafka topic.
2. Reads that stream in Real-time Mode, applies **stateless** transformations
   (`from_json`, `filter`, `withColumn`, `to_json`).
3. Sinks the result back to Kafka (`events-output`).

**Real-time Mode constraints (Spark 4.1.2):** stateless / map-like queries only,
output mode must be `update`, a `checkpointLocation` is required, and the trigger
duration is a *checkpoint interval* (min 5s), not a latency target. AQE is not
supported. It needs **one core per input partition** running continuously — the
input topic has 6 partitions and the cluster has 8 worker cores, so we are fine.


## 1. Configuration


In [ ]:
import os

KAFKA_BOOTSTRAP = os.environ.get("KAFKA_BOOTSTRAP", "kafka:9092")
INPUT_TOPIC     = "events-input"
OUTPUT_TOPIC    = "events-output"

# Stateless query: only the driver writes the checkpoint (offsets/commits),
# so a driver-local path is fine for this demo. Use S3/HDFS for stateful or
# production workloads.
CHECKPOINT_DIR  = "/home/jovyan/work/chk/realtime-demo"

print("bootstrap :", KAFKA_BOOTSTRAP)
print("input     :", INPUT_TOPIC)
print("output    :", OUTPUT_TOPIC)
print("checkpoint:", CHECKPOINT_DIR)


## 2. Make sure the topics exist

`docker compose` already runs a `kafka-init` job that creates both topics. This
cell is an idempotent safety net in case you are running the broker elsewhere.


In [ ]:
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

admin = KafkaAdminClient(bootstrap_servers=KAFKA_BOOTSTRAP)
for topic in (INPUT_TOPIC, OUTPUT_TOPIC):
    try:
        admin.create_topics([NewTopic(name=topic, num_partitions=6, replication_factor=1)])
        print("created", topic)
    except TopicAlreadyExistsError:
        print("exists ", topic)
admin.close()


## 3. Start the event producer (~10,000 events/sec)

We launch an inline producer in a background thread so the whole demo runs from
this single notebook. The same logic is available as a standalone script for
running in a terminal:

```bash
python /home/jovyan/work/kafka_producer.py --rate 10000
```


In [10]:
import threading, time, json, random
from kafka import KafkaProducer

EVENT_TYPES = ["click", "view", "purchase", "add_to_cart", "logout", "login"]
COUNTRIES   = ["US", "ES", "DE", "FR", "BR", "IN", "JP", "GB"]

def run_producer(rate=10000, duration=300):
    """Inline twin of kafka_producer.py so the notebook is self-contained."""
    producer = KafkaProducer(
        bootstrap_servers=KAFKA_BOOTSTRAP,
        acks=1, linger_ms=20, batch_size=64 * 1024,
        compression_type="lz4", buffer_memory=128 * 1024 * 1024,
        value_serializer=lambda v: json.dumps(v).encode("utf-8"),
        key_serializer=lambda k: k.encode("utf-8"),
    )
    window, per_window = 0.1, max(1, int(rate * 0.1))
    seq = sent = 0
    start = last = time.time()
    try:
        while True:
            ws = time.time()
            if duration > 0 and (ws - start) >= duration:
                break
            for _ in range(per_window):
                uid = random.randint(1, 100_000)
                evt = {"event_id": seq, "user_id": uid,
                       "event_type": random.choice(EVENT_TYPES),
                       "country": random.choice(COUNTRIES),
                       "amount": round(random.uniform(0.0, 500.0), 2),
                       "ts": time.time()}
                producer.send(INPUT_TOPIC, key=str(uid), value=evt)
                seq += 1; sent += 1
            now = time.time()
            if now - last >= 2.0:
                print(f"  produced={sent:,}  avg_rate={sent/(now-start):,.0f}/s")
                last = now
            spent = time.time() - ws
            if spent < window:
                time.sleep(window - spent)
    finally:
        producer.flush(); producer.close()
        print(f"Producer done. total={sent:,}")

producer_thread = threading.Thread(target=run_producer, kwargs={"rate": 10000, "duration": 300}, daemon=True)
producer_thread.start()
print("Producer thread started (~10k events/sec for 300s).")


Producer thread started (~10k events/sec for 300s).
  produced=20,000  avg_rate=9,659/s
  produced=39,000  avg_rate=9,485/s
  produced=60,000  avg_rate=9,600/s
  produced=80,000  avg_rate=9,697/s
  produced=100,000  avg_rate=9,753/s
  produced=120,000  avg_rate=9,777/s
  produced=140,000  avg_rate=9,768/s
  produced=157,000  avg_rate=9,571/s
  produced=177,000  avg_rate=9,573/s
  produced=198,000  avg_rate=9,631/s
  produced=217,000  avg_rate=9,619/s
  produced=237,000  avg_rate=9,642/s
  produced=254,000  avg_rate=9,546/s
  produced=275,000  avg_rate=9,580/s
  produced=295,000  avg_rate=9,597/s
  produced=315,000  avg_rate=9,606/s
  produced=335,000  avg_rate=9,614/s
  produced=355,000  avg_rate=9,625/s
  produced=375,000  avg_rate=9,628/s
  produced=395,000  avg_rate=9,643/s
  produced=412,000  avg_rate=9,581/s
  produced=431,000  avg_rate=9,567/s
  produced=451,000  avg_rate=9,577/s
  produced=471,000  avg_rate=9,588/s
  produced=491,000  avg_rate=9,598/s
  produced=511,000  avg_rat

## 4. Start the Spark session


In [11]:
import os
# belt-and-suspenders: clear any insecure Py4J hints
os.environ.pop("PYSPARK_GATEWAY_PORT", None)
os.environ.pop("PYSPARK_GATEWAY_SECRET", None)

try:
    spark.stop()
except Exception:
    pass

from pyspark.sql import SparkSession

# The Kafka connector jars (spark-sql-kafka-0-10, token provider, kafka-clients,
# commons-pool2) are baked into the Spark/Jupyter images, so no spark.jars.packages
# / Ivy resolution is needed here.
spark = (
    SparkSession.builder
      .master("spark://spark-master:7077")
      .appName("spark41-realtime-kafka")
      # Real-time Mode does not support Adaptive Query Execution.
      .config("spark.sql.adaptive.enabled", "false")
      .config("spark.dynamicAllocation.enabled", "false")
      .config("spark.eventLog.enabled", "false")
      .config("spark.sql.shuffle.partitions", "6")
      .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)


Spark version: 4.1.2


## 5. Define the stateless transformation

Parse the JSON payload, keep only revenue-bearing events, and enrich each record
with a couple of derived columns. Every operator here is map-like, so the query is
eligible for Real-time Mode.


In [12]:
from pyspark.sql.functions import col, from_json, to_json, struct, upper, current_timestamp, lit
from pyspark.sql.types import StructType, StructField, LongType, StringType, DoubleType

event_schema = StructType([
    StructField("event_id",   LongType()),
    StructField("user_id",    LongType()),
    StructField("event_type", StringType()),
    StructField("country",    StringType()),
    StructField("amount",     DoubleType()),
    StructField("ts",         DoubleType()),
])

raw = (
    spark.readStream
      .format("kafka")
      .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
      .option("subscribe", INPUT_TOPIC)
      .option("startingOffsets", "latest")
      .load()
)

parsed = (
    raw.selectExpr("CAST(value AS STRING) AS json_str")
       .select(from_json(col("json_str"), event_schema).alias("e"))
       .select("e.*")
)

# --- stateless transformations ---
transformed = (
    parsed
      .filter((col("event_type").isin("purchase", "add_to_cart")) & (col("amount") > 0))
      .withColumn("amount_with_tax", (col("amount") * lit(1.21)))
      .withColumn("country", upper(col("country")))
      .withColumn("processed_at", current_timestamp())
)

# Build Kafka key/value (value is JSON string).
out = transformed.select(
    col("user_id").cast("string").alias("key"),
    to_json(struct(
        "event_id", "user_id", "event_type", "country",
        "amount", "amount_with_tax", "processed_at"
    )).alias("value"),
)


## 6. Write back to Kafka in Real-time Mode

Real-time Mode is enabled by the JVM trigger `Trigger.RealTime("...")`. As of Spark
4.1.2 the PySpark `trigger()` wrapper does not expose `realTime`, so we attach the JVM
trigger to the underlying Java writer via py4j. It is combined with
`.outputMode("update")`. The `"10 seconds"` is the **checkpoint cadence**, not a latency
target; records are still emitted within milliseconds.

In [13]:
# PySpark 4.1.2 exposes processingTime/once/continuous/availableNow on trigger(),
# but NOT the new Real-time Mode trigger. The JVM has it (Trigger.RealTime), so we
# call it through py4j and attach it to the underlying Java DataStreamWriter.
writer = (
    out.writeStream
      .format("kafka")
      .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
      .option("topic", OUTPUT_TOPIC)
      .option("checkpointLocation", CHECKPOINT_DIR)
      .outputMode("update")            # required by Real-time Mode
)

# Trigger.RealTime("<batch duration>") -> the duration is a CHECKPOINT interval
# (min 5s), not a latency target. Records still flow within milliseconds.
_jvm = spark._sc._jvm
_real_time_trigger = _jvm.org.apache.spark.sql.streaming.Trigger.RealTime("10 seconds")
writer._jwrite.trigger(_real_time_trigger)

query = writer.start()
print("Streaming query started:", query.id)
print("status:", query.status)

Streaming query started: 4075b2a7-5ced-4278-85a1-01dafb407319
status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


26/06/23 09:46:13 WARN MicroBatchExecution: Disabling AQE since AQE is not supported for Real-time Mode.


## 7. Verify the output stream

Consume a few transformed records straight off `events-output` with a plain Kafka
consumer to confirm the pipeline end to end.


In [14]:
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    OUTPUT_TOPIC,
    bootstrap_servers=KAFKA_BOOTSTRAP,
    auto_offset_reset="latest",
    consumer_timeout_ms=15000,
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
)

seen = 0
for msg in consumer:
    print(msg.value)
    seen += 1
    if seen >= 10:
        break
consumer.close()
print(f"\nRead {seen} transformed events from {OUTPUT_TOPIC}.")


[Stage 0:=================================================>         (5 + 1) / 6]

{'event_id': 1, 'user_id': 42134, 'event_type': 'purchase', 'country': 'IN', 'amount': 263.32, 'amount_with_tax': 318.61719999999997, 'processed_at': '2026-06-23T09:46:13.846Z'}


[Stage 1:>                                                          (0 + 6) / 6]

{'event_id': 31, 'user_id': 85589, 'event_type': 'add_to_cart', 'country': 'DE', 'amount': 182.57, 'amount_with_tax': 220.9097, 'processed_at': '2026-06-23T09:46:27.779Z'}
{'event_id': 37, 'user_id': 82017, 'event_type': 'purchase', 'country': 'FR', 'amount': 273.42, 'amount_with_tax': 330.83820000000003, 'processed_at': '2026-06-23T09:46:27.779Z'}
{'event_id': 62, 'user_id': 84424, 'event_type': 'add_to_cart', 'country': 'BR', 'amount': 299.67, 'amount_with_tax': 362.6007, 'processed_at': '2026-06-23T09:46:27.779Z'}
{'event_id': 71, 'user_id': 17358, 'event_type': 'add_to_cart', 'country': 'GB', 'amount': 405.09, 'amount_with_tax': 490.15889999999996, 'processed_at': '2026-06-23T09:46:27.779Z'}
{'event_id': 101, 'user_id': 72320, 'event_type': 'add_to_cart', 'country': 'FR', 'amount': 426.56, 'amount_with_tax': 516.1376, 'processed_at': '2026-06-23T09:46:27.779Z'}
{'event_id': 104, 'user_id': 82608, 'event_type': 'add_to_cart', 'country': 'US', 'amount': 396.52, 'amount_with_tax': 479

## 8. Live progress (input vs processing rate)


In [15]:
import time
for _ in range(5):
    p = query.lastProgress
    if p:
        print(f"batch={p.get('batchId')}  "
              f"inputRate={p.get('inputRowsPerSecond', 0):.0f}/s  "
              f"procRate={p.get('processedRowsPerSecond', 0):.0f}/s")
    time.sleep(5)


batch=51  inputRate=0/s  procRate=0/s
batch=51  inputRate=0/s  procRate=0/s


batch=52  inputRate=12045/s  procRate=15928/s


[Stage 2:>                                                          (0 + 6) / 6]

batch=52  inputRate=12045/s  procRate=15928/s


batch=53  inputRate=17937/s  procRate=18300/s


[Stage 3:>                                                          (0 + 6) / 6]

## 9. Measure end-to-end latency (produce -> RTM transform -> consume)

Section 8 shows the pipeline keeps up on *throughput*; it does not tell you how
*fast* a single record travels end to end. Because the query carries the original
`event_id` through to `events-output`, we can measure that directly: stamp a small
batch of **probe** events on the way in, recognize them on the way out by their
`event_id`, and record the wall-clock delta per record. The probe events use a high
`event_id` sentinel and `event_type="purchase"` / `amount>0` so they pass the
pipeline's filter and stand out from the ~10k/s background traffic. Read the result
as a distribution: `p50` is the typical record while `p95`/`p99` are the tail
latencies users actually feel. On this single-host demo expect a low-tens-of-ms
median with a fatter tail than a real multi-node cluster would show.


In [18]:
import time, json
from kafka import KafkaProducer, KafkaConsumer

PROBE_N        = 500
PROBE_SENTINEL = 9_000_000_000          # event_id base well above the producer's range
PROBE_IDS      = list(range(PROBE_SENTINEL, PROBE_SENTINEL + PROBE_N))

# Subscribe at 'latest' BEFORE producing so we don't miss fast records, then poll
# once to force partition assignment.
probe_consumer = KafkaConsumer(
    OUTPUT_TOPIC,
    bootstrap_servers=KAFKA_BOOTSTRAP,
    auto_offset_reset="latest",
    consumer_timeout_ms=20000,
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
)
probe_consumer.poll(timeout_ms=1000)

probe_producer = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP,
    acks=1,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: k.encode("utf-8"),
)

# Stamp the send time locally, keyed by event_id (which survives the transform).
send_ts = {}
for eid in PROBE_IDS:
    uid = eid % 100_000
    evt = {"event_id": eid, "user_id": uid, "event_type": "purchase",
           "country": "ES", "amount": 42.0, "ts": time.time()}
    send_ts[eid] = time.time()
    probe_producer.send(INPUT_TOPIC, key=str(uid), value=evt)
probe_producer.flush()

# Match probe events on the output and record end-to-end latency (ms).
latencies_ms = []
seen_ids = set()
for msg in probe_consumer:
    eid = msg.value.get("event_id")
    if eid in send_ts and eid not in seen_ids:
        latencies_ms.append((time.time() - send_ts[eid]) * 1000.0)
        seen_ids.add(eid)
        if len(seen_ids) >= PROBE_N:
            break
probe_consumer.close()
probe_producer.close()

if latencies_ms:
    latencies_ms.sort()
    def pct(p):
        k = min(len(latencies_ms) - 1, int(round((p / 100.0) * (len(latencies_ms) - 1))))
        return latencies_ms[k]
    print(f"matched {len(latencies_ms)}/{PROBE_N} probe events")
    print(f"end-to-end (produce -> consume)  "
          f"p50={pct(50):7.1f}  p95={pct(95):7.1f}  "
          f"p99={pct(99):7.1f}  max={max(latencies_ms):7.1f}  (ms)")
else:
    print("No probe events matched. Is the streaming query running? "
          "Increase consumer_timeout_ms or PROBE_N and retry.")


matched 500/500 probe events
end-to-end (produce -> consume)  p50=  717.4  p95=  896.6  p99=  899.1  max= 1047.1  (ms)


[Stage 15:>                                                         (0 + 6) / 6]

## 9. Stop the query


In [9]:
query.stop()
print("Query stopped. The producer thread is a daemon and stops with the kernel.")


Query stopped. The producer thread is a daemon and stops with the kernel.


26/06/23 09:46:03 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 51, writer: org.apache.spark.sql.kafka010.KafkaStreamingWrite@3788c7c7] is aborting.
26/06/23 09:46:03 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 51, writer: org.apache.spark.sql.kafka010.KafkaStreamingWrite@3788c7c7] aborted.
26/06/23 09:46:03 WARN DAGScheduler: Failed to cancel job group b91c9af6-91b1-43d9-92c4-54fc72d76ada. Cannot find active jobs for it.
26/06/23 09:46:03 WARN TaskSetManager: Lost task 4.0 in stage 44.0 (TID 268) (172.20.0.7 executor 0): TaskKilled (Stage cancelled: [SPARK_JOB_CANCELLED] Job 44 cancelled Query [id = 4075b2a7-5ced-4278-85a1-01dafb407319, runId = b91c9af6-91b1-43d9-92c4-54fc72d76ada] was stopped SQLSTATE: XXKDA)
26/06/23 09:46:03 WARN TaskSetManager: Lost task 2.0 in stage 44.0 (TID 266) (172.20.0.7 executor 0): TaskKilled (Stage cancelled: [SPARK_JOB_CANCELLED] Job 44 cancelled Query [id = 4075b2a7-5ced-4278-85a1-01d